# CRISP-DM Stage 5: Deployment

**Cloudflare D1 Database SQL Seed Generator**

In [ ]:
import os, json, pandas as pd
from config import CLEANED_PROVINCES_CSV, CLUSTERED_REGENCIES_CSV, EVALUATE_METRICS_JSON, INTERPRET_REPORT_MD, SEED_SQL

def sql_val(val):
    if pd.isna(val) or val is None:
        return "NULL"
    if isinstance(val, (int, float)):
        return str(val)
    escaped = str(val).replace("'", "''")
    return f"'{escaped}'"

df_p = pd.read_csv(CLEANED_PROVINCES_CSV)
df_r = pd.read_csv(CLUSTERED_REGENCIES_CSV)

with open(EVALUATE_METRICS_JSON, encoding='utf-8') as f:
    metrics_data = json.load(f)

ai_report = ""
if os.path.exists(INTERPRET_REPORT_MD):
    with open(INTERPRET_REPORT_MD, encoding='utf-8') as f:
        ai_report = f.read()

sql_lines = ["-- Cloudflare D1 SQL Seed Generated Automatically by Pipeline\n", "-- Insert Provinces"]
for _, r in df_p.iterrows():
    p_id = sql_val(r.get('province_id', r.get('no')))
    p_name = sql_val(r['province_name'])
    lat = sql_val(r.get('latitude'))
    lon = sql_val(r.get('longitude'))
    sql_lines.append(f"INSERT OR REPLACE INTO provinces (id, name, total_koperasi, koperasi_nib, koperasi_npwp, koperasi_rat, simpanan_pokok, simpanan_wajib, volume_transaksi, nilai_transaksi, latitude, longitude, rasio_nib, rasio_npwp, rasio_rat) VALUES ({p_id}, {p_name}, {r['total_koperasi']}, {r['koperasi_nib']}, {r['koperasi_npwp']}, {r['koperasi_rat']}, {r['simpanan_pokok']}, {r['simpanan_wajib']}, {r['volume_transaksi']}, {r['nilai_transaksi']}, {lat}, {lon}, {r['rasio_nib']}, {r['rasio_npwp']}, {r['rasio_rat']});")

sql_lines.append("\n-- Insert Regencies")
for _, r in df_r.iterrows():
    r_id = sql_val(f"{r['province_id']}_{r['regency_no']}")
    r_name = sql_val(r['regency_name'])
    lat = sql_val(r.get('latitude'))
    lon = sql_val(r.get('longitude'))
    sql_lines.append(f"INSERT OR REPLACE INTO regencies (id, province_id, name, total_koperasi, koperasi_nib, koperasi_npwp, koperasi_rat, simpanan_pokok, simpanan_wajib, volume_transaksi, nilai_transaksi, latitude, longitude, rasio_nib, rasio_npwp, rasio_rat, cluster_label) VALUES ({r_id}, {r['province_id']}, {r_name}, {r['total_koperasi']}, {r['koperasi_nib']}, {r['koperasi_npwp']}, {r['koperasi_rat']}, {r['simpanan_pokok']}, {r['simpanan_wajib']}, {r['volume_transaksi']}, {r['nilai_transaksi']}, {lat}, {lon}, {r['rasio_nib']}, {r['rasio_npwp']}, {r['rasio_rat']}, {r['cluster_label']});")

sql_lines.append("\n-- Insert Evaluation Metrics")
m_json = sql_val(json.dumps(metrics_data))
a_rep = sql_val(ai_report)
sql_lines.append(f"INSERT OR REPLACE INTO pipeline_metrics (id, metrics_json, ai_report, created_at) VALUES (1, {m_json}, {a_rep}, CURRENT_TIMESTAMP);")

os.makedirs(os.path.dirname(SEED_SQL), exist_ok=True)
with open(SEED_SQL, 'w', encoding='utf-8') as f:
    f.write("\n".join(sql_lines))

print(f"Deployment Seed generated successfully at {SEED_SQL}")
